# ColBERTv2 vs Bi-Encoder Retrieval Benchmark

End-to-end pipeline: data loading → chunking → NER classification → bi-encoder retrieval → ColBERTv2 retrieval → evaluation → visualization.

All results are written to a single JSON log file. Evaluation and visualization read exclusively from that log.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import json
import yaml
import pandas as pd
from IPython.display import Image, display

from src.profiler import Profiler
from src.benchmark_runner import load_config, build_variant_config, run_full_benchmark

config = load_config("configs/quick_ablation.yaml")

print("Config loaded.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Default chunking strategy: {config['chunking']['strategy']}")


## 1. Data Pipeline

Load KILT NQ, build reduced corpus, chunk, classify queries by NER, sample balanced groups, extract ground-truth.

In [ ]:
from src.data_pipeline import run_data_pipeline
import gc

profiler = Profiler(config=config)
pipeline_out = run_data_pipeline(config, profiler)

chunks = pipeline_out["chunks"]
sampled_queries = pipeline_out["sampled_queries"]
del pipeline_out
gc.collect()

print(f"Total chunks: {len(chunks)}")
print(f"Sampled queries: {len(sampled_queries)}")
print(f"  single-entity: {sum(1 for q in sampled_queries if q['entity_group'] == 'single-entity')}")
print(f"  multi-entity:  {sum(1 for q in sampled_queries if q['entity_group'] == 'multi-entity')}")

## 2. Bi-Encoder Retrieval

Encode chunks with MiniLM, build FAISS index, retrieve top-k for each query.

In [ ]:
from src.biencoder_retrieval import run_biencoder_retrieval

run_biencoder_retrieval(chunks, sampled_queries, config, profiler)
print("Bi-encoder retrieval complete.")
print(f"  Sample query recall@10: {sampled_queries[0].get('biencoder_recall_at_k', {}).get('10', 'N/A')}")

## 3. ColBERTv2 Retrieval

Index chunks with RAGatouille ColBERTv2, retrieve top-k with late interaction scoring.

In [ ]:
from src.colbert_retrieval import run_colbert_retrieval

run_colbert_retrieval(chunks, sampled_queries, config, profiler)
print("ColBERTv2 retrieval complete.")
print(f"  Sample query recall@10: {sampled_queries[0].get('colbert_recall_at_k', {}).get('10', 'N/A')}")

## 4. Save JSON Log & Log Per-Query Results

Write per-query results to the profiler and save the complete JSON log.

In [ ]:
# Log per-query results into the profiler
for q in sampled_queries:
    record = {
        "query": q["query"],
        "entity_count": q["entity_count"],
        "entity_list": q["entity_list"],
        "entity_group": q["entity_group"],
        "ground_truth_chunk_ids": q["ground_truth_chunk_ids"],
    }
    # Bi-encoder results
    for key in ("biencoder_retrieved_ids", "biencoder_recall_at_k", "biencoder_latency_ms"):
        if key in q:
            record[key] = q[key]
    # ColBERT results
    for key in ("colbert_retrieved_ids", "colbert_recall_at_k", "colbert_latency_ms"):
        if key in q:
            record[key] = q[key]
    profiler.log_query(record)

# Store model names in metadata
profiler.data["metadata"]["models"] = config["models"]
profiler.data["metadata"]["k_values"] = config["retrieval"]["k_values"]

# Save JSON log
log_path = config["paths"]["json_log"]
profiler.save(log_path)
print(f"JSON log saved to: {log_path}")

# Verify key fields
import json
with open(log_path, "r", encoding="utf-8") as f:
    log = json.load(f)
print(f"  Stages recorded: {list(log['stages'].keys())}")
print(f"  Queries logged: {len(log['queries'])}")
print(f"  Disk sizes: {log['disk_sizes']}")
print(f"  Metadata keys: {list(log['metadata'].keys())}")

## 5. Evaluation

Compute Recall@k from the JSON log (reads exclusively from JSON, no recomputation).

In [ ]:
from src.evaluation import run_evaluation

summary = run_evaluation(config["paths"]["json_log"])
print("Summary Statistics:")
print(summary.to_string(index=False))

## 6. Visualization

Generate all charts and CSV export (reads exclusively from JSON log).

In [ ]:
from src.visualize import run_visualization

output_paths = run_visualization(
    log_path=config["paths"]["json_log"],
    charts_dir=config["paths"]["charts_dir"],
    csv_path=config["paths"]["csv_output"],
)

print("Generated outputs:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")

## 7. Display Charts

In [ ]:
from IPython.display import Image, display

for name, path in output_paths.items():
    if isinstance(path, str) and path.endswith(".png"):
        print(f"\n--- {name} ---")
        display(Image(filename=path))
    elif isinstance(path, list):
        for p in path:
            print(f"\n--- {p} ---")
            display(Image(filename=p))

## 8. Profiling Summary

In [ ]:
import json

with open(config["paths"]["json_log"], encoding="utf-8") as f:
    log = json.load(f)

print("=== Stage Timings ===")
for stage, info in log["stages"].items():
    vram = f", VRAM peak: {info['peak_vram_bytes']/1e9:.2f} GB" if info.get("peak_vram_bytes") else ""
    print(f"  {stage}: {info['duration_seconds']:.2f}s, RSS: {info['rss_bytes']/1e9:.2f} GB{vram}")

print(f"\n=== Disk Sizes ===")
for label, size in log["disk_sizes"].items():
    print(f"  {label}: {size/1e6:.1f} MB")

print(f"\n=== Run Metadata ===")
meta = log["metadata"]
print(f"  Timestamp: {meta['timestamp']}")
print(f"  GPU: {meta.get('gpu_device', 'None')}")
print(f"  Corpus size: {meta.get('corpus_size', 'N/A')} pages")
print(f"  Total chunks: {meta.get('total_chunks', 'N/A')}")
print(f"  Queries: {meta.get('total_queries', 'N/A')}")
print(f"  Queries per group: {meta.get('queries_per_group', 'N/A')}")

## 9. Strategy Sections

The sections below run one chunking strategy at a time and save outputs under `results/notebook/<variant>/`.
Each section is intentionally explicit so the workflow is easy to follow in presentation and debugging.


In [ ]:
NOTEBOOK_VARIANTS = [
    "paragraph",
    "sentence_window",
    "adaptive_sentence",
    "semantic_similarity",
    "adaptive_sentence_keyword",
]

def show_saved_variant_outputs(label: str):
    summary_path = Path("results") / "notebook" / label / "summary_statistics.csv"
    charts_dir = Path("results") / "notebook" / label / "charts"

    if summary_path.exists():
        summary_df = pd.read_csv(summary_path)
        display(summary_df)
    else:
        print(f"No summary CSV found for {label}.")

    for image_name in ["recall_bar_chart.png", "latency_comparison.png", "index_size_comparison.png"]:
        image_path = charts_dir / image_name
        if image_path.exists():
            print(f"\n--- {label}: {image_name} ---")
            display(Image(filename=str(image_path)))


## 10. Sentence Window Benchmark


In [ ]:
sentence_window_config = build_variant_config(
    config,
    run_subdir="notebook/sentence_window",
    overrides={
        "chunking": {
            "strategy": "sentence_window",
            "sentence_window_size": 3,
            "sentence_window_stride": 2,
        }
    },
    cache_suffix="sentence_window",
)

sentence_window_result = run_full_benchmark(sentence_window_config)
print(sentence_window_result["summary"].to_string(index=False))


## 11. Adaptive Sentence Benchmark


In [ ]:
adaptive_sentence_config = build_variant_config(
    config,
    run_subdir="notebook/adaptive_sentence",
    overrides={
        "chunking": {
            "strategy": "adaptive_sentence",
            "adaptive_min_words": 60,
            "adaptive_max_words": 120,
        }
    },
    cache_suffix="adaptive_sentence",
)

adaptive_sentence_result = run_full_benchmark(adaptive_sentence_config)
print(adaptive_sentence_result["summary"].to_string(index=False))


## 12. Semantic Similarity Benchmark


In [ ]:
semantic_similarity_config = build_variant_config(
    config,
    run_subdir="notebook/semantic_similarity",
    overrides={
        "chunking": {
            "strategy": "semantic_similarity",
            "semantic_model": "sentence-transformers/all-MiniLM-L6-v2",
            "semantic_similarity_threshold": 0.72,
            "semantic_min_words": 60,
            "semantic_max_words": 120,
            "semantic_min_sentences": 2,
            "semantic_max_sentences": 5,
        }
    },
    cache_suffix="semantic_similarity",
)

semantic_similarity_result = run_full_benchmark(semantic_similarity_config)
print(semantic_similarity_result["summary"].to_string(index=False))


## 13. Enhanced Adaptive Sentence Benchmark


In [ ]:
adaptive_sentence_keyword_config = build_variant_config(
    config,
    run_subdir="notebook/adaptive_sentence_keyword",
    overrides={
        "chunking": {
            "strategy": "adaptive_sentence_keyword",
            "adaptive_min_words": 60,
            "adaptive_max_words": 120,
            "adaptive_keyword_slack_words": 40,
            "adaptive_keyword_min_overlap": 1,
        }
    },
    cache_suffix="adaptive_sentence_keyword",
)

adaptive_sentence_keyword_result = run_full_benchmark(adaptive_sentence_keyword_config)
print(adaptive_sentence_keyword_result["summary"].to_string(index=False))


## 14. Compare Variant Summaries


In [ ]:
variant_rows = []
for label in NOTEBOOK_VARIANTS:
    summary_path = Path("results") / "notebook" / label / "summary_statistics.csv"
    if not summary_path.exists():
        continue
    summary_df = pd.read_csv(summary_path)
    overall = summary_df[summary_df["entity_group"] == "overall"].copy()
    row = {"variant": label}
    for _, metric_row in overall.iterrows():
        k = int(metric_row["k"])
        row[f"biencoder_recall_at_{k}"] = metric_row["biencoder_recall"]
        row[f"colbert_recall_at_{k}"] = metric_row["colbert_recall"]
        row[f"delta_at_{k}"] = metric_row["delta"]
    variant_rows.append(row)

variant_overview = pd.DataFrame(variant_rows)
variant_overview


## 15. Display Saved Results for Any Variant

Set `SELECTED_VARIANT` to inspect any completed run without rerunning the strategy cell.


In [ ]:
SELECTED_VARIANT = "adaptive_sentence"
show_saved_variant_outputs(SELECTED_VARIANT)


## 16. Error Analysis

Compare two completed variants at query level. This section is intended for focused qualitative analysis rather than rerunning the whole benchmark.


In [ ]:
from src.error_analysis import compare_variant_logs, summarize_comparison

BASELINE_VARIANT = "paragraph"
CANDIDATE_VARIANT = "adaptive_sentence_keyword"
ERROR_ANALYSIS_RETRIEVER = "colbert"
ERROR_ANALYSIS_K = 10

comparison = compare_variant_logs(
    baseline_log_path=str(Path("results") / "notebook" / BASELINE_VARIANT / "benchmark_log.json"),
    candidate_log_path=str(Path("results") / "notebook" / CANDIDATE_VARIANT / "benchmark_log.json"),
    retriever=ERROR_ANALYSIS_RETRIEVER,
    k=ERROR_ANALYSIS_K,
)

print(summarize_comparison(comparison).to_string(index=False))
comparison.head(10)


## 17. Knowledge Graph Prototype

Build and evaluate the lightweight entity graph baseline using the current config.


In [ ]:
from src.knowledge_graph import build_graph_from_config, graph_summary_dataframe, run_graph_retrieval, save_graph

kg_config = load_config("configs/quick_ablation.yaml")
kg_output_dir = Path("results") / "notebook" / "knowledge_graph"
kg_output_dir.mkdir(parents=True, exist_ok=True)

kg_built = build_graph_from_config(kg_config)
graph = kg_built["graph"]
kg_queries = kg_built["sampled_queries"]

save_graph(graph, str(kg_output_dir / "knowledge_graph.json"))
run_graph_retrieval(
    graph=graph,
    sampled_queries=kg_queries,
    k_values=kg_config["retrieval"]["k_values"],
    spacy_model=kg_config["spacy"]["model"],
)

graph_summary = graph_summary_dataframe(kg_queries)
graph_summary.to_csv(kg_output_dir / "graph_recall_summary.csv", index=False)
print(graph_summary.to_string(index=False))


## 18. Project Comparison Charts

Build a compact comparison table from completed notebook variants and visualize the key trends.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

variant_rows = []
for label in NOTEBOOK_VARIANTS:
    summary_path = Path("results") / "notebook" / label / "summary_statistics.csv"
    if not summary_path.exists():
        continue
    summary_df = pd.read_csv(summary_path)
    overall = summary_df[summary_df["entity_group"] == "overall"].copy()
    row = {"variant": label}
    for _, metric_row in overall.iterrows():
        k = int(metric_row["k"])
        row[f"biencoder_recall_at_{k}"] = metric_row["biencoder_recall"]
        row[f"colbert_recall_at_{k}"] = metric_row["colbert_recall"]
        row[f"delta_at_{k}"] = metric_row["delta"]
    variant_rows.append(row)

variant_overview = pd.DataFrame(variant_rows)
variant_overview


In [ ]:
if not variant_overview.empty:
    chart_df = variant_overview.melt(
        id_vars=["variant"],
        value_vars=["colbert_recall_at_1", "colbert_recall_at_5", "colbert_recall_at_10", "colbert_recall_at_20"],
        var_name="metric",
        value_name="recall",
    )
    chart_df["k"] = chart_df["metric"].str.extract(r"(\d+)").astype(int)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=chart_df, x="k", y="recall", hue="variant", marker="o", ax=ax)
    ax.set_title("ColBERT Recall@k by Chunking Strategy")
    ax.set_xlabel("k")
    ax.set_ylabel("Recall@k")
    plt.show()
else:
    print("No completed variant summaries found yet.")


## 19. Optional Cleanup Notes

If a heavy section fails partway through, keep the partial artifacts for inspection and rerun just that section later instead of restarting the full notebook.
